# CO₂: watch vibrational energy move
Use **ASE + MACE-OFF23-small** for three atoms. Stretch both C–O bonds and release from rest.
Plot bond lengths, kinetic energy and potential energy. No thermostat is used: it would
exchange energy with the vibration we want to study. This is classical motion, not
a quantum vibrational state or an infrared intensity calculation.

**Model choice.** [MACE-OFF23-small](https://github.com/ACEsuit/mace) is a neutral-molecule model covering C and O. We use the small version for fast CPU force evaluations in this short vibration demo.

Complete the `...` placeholders in **Your turn** cells. Other cells are provided.


In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from ase import Atoms, units
from ase.build import molecule
from ase.io import read, write
from ase.optimize import BFGS
from ase.md.langevin import Langevin
from ase.md.verlet import VelocityVerlet
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary

sys.path.insert(0, str(Path.cwd().parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.view_mols import view
_, OUTPUT = start_exercise()
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '1')))

from mace.calculators import MACECalculator
calc = MACECalculator(model_paths='../models/MACE-OFF23_small.model',
                 device='cpu', default_dtype='float64')


## 1. Relax, then excite a symmetric stretch
ASE orders this molecule as C, O, O. Extending both bonds by 4% creates a small vibrational displacement.

**Your turn.** Attach the supplied MLIP calculator. Relaxation and the initial displacement are provided.


In [ ]:
atoms = molecule('CO2')
atoms.info.update(total_charge=0, total_spin=0)
atoms.calc = ...  # TODO: attach the calculator
assert BFGS(atoms, logfile=str(OUTPUT/'relax.log')).run(fmax=0.01, steps=100)
atoms.positions[1:] = atoms.positions[0] + 1.04*(atoms.positions[1:]-atoms.positions[0])
atoms.set_momenta(np.zeros((len(atoms),3)))
view(atoms)


## 2. Run NVE
Use a 0.5 fs timestep and 250 steps (125 fs). Check energy conservation for this model; repeat at half the timestep to assess integration error.

**Your turn.** Complete VelocityVerlet with the ASE structure and timestep in ASE units, then run STEPS integration steps. Trajectory writing is provided.


In [ ]:
DT_FS = 0.5
STEPS = 250
dyn = VelocityVerlet(..., ..., 
                     trajectory=str(OUTPUT/'co2.traj'), loginterval=2)
start = time.perf_counter()
...  # TODO: run STEPS integration steps
print(f'Elapsed: {(time.perf_counter()-start)/60:.1f} min')


## 3. Inspect the energy exchange
Potential and kinetic energy oscillate while their sum should remain approximately constant.

In [ ]:
frames = read(OUTPUT/'co2.traj', index=':')
t = np.arange(len(frames))*2*DT_FS
potential = np.array([a.get_potential_energy() for a in frames])
kinetic = np.array([a.get_kinetic_energy() for a in frames])
fig, ax = plt.subplots(2,1,figsize=(7,5),sharex=True)
for oxygen in [1,2]: ax[0].plot(t,[a.get_distance(0,oxygen) for a in frames],label=f'C–O{oxygen}')
ax[0].set_ylabel('Bond length (Å)'); ax[0].legend()
ax[1].plot(t,potential-potential.min(),label='Potential (shifted)')
ax[1].plot(t,kinetic,label='Kinetic')
ax[1].plot(t,potential+kinetic-potential.min(),label='Total (same shift)')
ax[1].set(xlabel='Time (fs)',ylabel='Energy (eV)'); ax[1].legend()
plt.tight_layout(); plt.show()
view(frames[::3], animate=True)


## Try and explain
Estimate the oscillation period from the bond-length plot. Repeat with half the timestep
and twice as many steps. What changes if only one C–O bond is stretched? Two coincident
bond traces are expected for the symmetric initial displacement.